# 04 — Country Benchmarking and Energy Mix

This notebook compares country candidates with transparent coverage rules. It separates absolute consumption, per-capita indicators, shares, and carbon intensity.

> **Reading guide:** Each analysis section follows the same pattern: question → method → code → interpretation. The conclusion directly below each major code block explains what the output means and what it does *not* establish.

## 1. Define a comparable country sample

**Question.** Which entity-year records can be used for country benchmarking?

I use non-null ISO codes to identify country candidates and exclude the `World` aggregate explicitly. This is a pragmatic filter, not a perfect sovereignty classification; source-specific entities without ISO codes are not included in country rankings.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "owid_energy.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "owid_energy.csv"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["savefig.dpi"] = 160

raw = pd.read_csv(DATA_PATH, low_memory=False)
df = raw.copy()
print(f"Loaded {DATA_PATH.name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

country_df = df.loc[df["iso_code"].notna() & ~df["country"].eq("World")].copy()
country_df["year"] = pd.to_numeric(country_df["year"], errors="coerce")
print(f"Country-candidate observations: {len(country_df):,}")
print(f"Country-candidate entity labels: {country_df['country'].nunique():,}")
print(f"Year range: {int(country_df['year'].min())}–{int(country_df['year'].max())}")

Loaded owid_energy.csv: 19,494 rows × 130 columns
Country-candidate observations: 17,480
Country-candidate entity labels: 220
Year range: 1700–2025


### Conclusion

ISO-based filtering makes the ranking sample more consistent, but it can exclude entities without ISO codes and may include territories depending on the source. I report the sample definition and year with each ranking, and I do not interpret the result as a definitive ranking of sovereign states.

## 2. Latest-year coverage check

**Question.** Which recent year offers a reasonable basis for comparing countries?

I inspect country coverage for the core indicators across recent years. A year is selected separately for each metric based on the number of valid observations; no universal minimum is imposed without seeing the coverage.

In [2]:
bench_metrics = [
    "total_energy_twh", "total_energy_per_capita_kwh", "total_energy_per_gdp_kwh_per_dollar",
    "renewables_electricity_share_pct", "low_carbon_electricity_share_pct",
    "fossil_fuels_electricity_share_pct", "solar_electricity_twh", "wind_electricity_twh",
    "electricity_carbon_intensity_gco2eq_per_kwh"
]
bench_metrics = [c for c in bench_metrics if c in country_df.columns]
coverage_rows = []
for year, g in country_df.groupby("year"):
    if year < 2015: continue
    for col in bench_metrics:
        coverage_rows.append({"year": int(year), "indicator": col,
                              "valid_entities": int(g[col].notna().sum()),
                              "entities_total": int(g["country"].nunique()),
                              "coverage_pct": round(g[col].notna().mean()*100, 2)})
benchmark_coverage = pd.DataFrame(coverage_rows)
benchmark_coverage.to_csv(TABLE_DIR / "country_benchmark_coverage_recent.csv", index=False)
display(benchmark_coverage.pivot(index="year", columns="indicator", values="valid_entities").tail(10))

indicator,electricity_carbon_intensity_gco2eq_per_kwh,fossil_fuels_electricity_share_pct,low_carbon_electricity_share_pct,renewables_electricity_share_pct,solar_electricity_twh,total_energy_per_capita_kwh,total_energy_per_gdp_kwh_per_dollar,total_energy_twh,wind_electricity_twh
year,,,,,,,,,
2016,212,212,212,212,213,216,165,218,208
2017,212,212,212,212,213,216,165,218,209
2018,212,212,212,212,213,216,165,218,209
2019,212,212,212,212,213,216,165,218,209
2020,212,212,212,212,213,216,165,218,210
2021,212,212,212,212,213,216,165,218,210
2022,212,212,212,212,213,216,165,218,210
2023,210,210,210,210,212,216,0,218,209
2024,195,195,195,195,196,216,0,218,194


### Conclusion

The best comparison year depends on the indicator. A later year with fewer reporting entities may still be useful for one variable but unsuitable for another. Rankings below use the latest year with a transparent minimum coverage threshold, and the selected year and sample size are included in the output table.

## 3. Country rankings

**Question.** Which country candidates have the highest values for selected indicators?

For each indicator, I choose the latest year with at least 30 valid country-candidate observations. This threshold is an explicit project rule, not a universal statistical standard. It can be adjusted in sensitivity analysis.

In [3]:
ranking_rows = []
for metric in bench_metrics:
    coverage = benchmark_coverage.loc[benchmark_coverage["indicator"].eq(metric)]
    eligible_years = coverage.loc[coverage["valid_entities"] >= 30, "year"]
    if eligible_years.empty: continue
    selected_year = int(eligible_years.max())
    subset = country_df.loc[country_df["year"].eq(selected_year), ["country", "iso_code", metric]].dropna()
    if len(subset) < 30: continue
    ascending = metric in ["electricity_carbon_intensity_gco2eq_per_kwh", "total_energy_per_gdp_kwh_per_dollar"]
    subset = subset.sort_values(metric, ascending=ascending).head(10)
    direction = "lowest" if ascending else "highest"
    for rank, (_, row) in enumerate(subset.iterrows(), start=1):
        ranking_rows.append({"indicator": metric, "year": selected_year, "direction": direction,
                             "rank": rank, "country": row["country"], "iso_code": row["iso_code"],
                             "value": row[metric], "valid_country_candidates": int(country_df.loc[country_df["year"].eq(selected_year), metric].notna().sum())})
rankings = pd.DataFrame(ranking_rows)
rankings.to_csv(TABLE_DIR / "country_top10_benchmarks.csv", index=False)
display(rankings.head(30))

,indicator,year,direction,rank,country,iso_code,value,valid_country_candidates
0,total_energy_twh,2025,highest,1,China,CHN,45053.710000,79
1,total_energy_twh,2025,highest,2,United States,USA,26063.510000,79
2,total_energy_twh,2025,highest,3,India,IND,10860.910000,79
3,total_energy_twh,2025,highest,4,Russia,RUS,8663.982000,79
4,total_energy_twh,2025,highest,5,Japan,JPN,4578.486000,79
5,total_energy_twh,2025,highest,6,Iran,IRN,3688.026100,79
6,total_energy_twh,2025,highest,7,South Korea,KOR,3603.519800,79
7,total_energy_twh,2025,highest,8,Canada,CAN,3348.667000,79
8,total_energy_twh,2025,highest,9,Saudi Arabia,SAU,3340.263400,79
9,total_energy_twh,2025,highest,10,Indonesia,IDN,3198.909400,79


### Conclusion

The rankings answer a narrow question: which included country candidates have the highest or lowest observed values for the selected metric and year? Total energy consumption tends to favor large economies, while per-capita indicators answer a different question. Rankings are descriptive, not judgments about policy performance or investment quality.

## 4. Recent multi-year averages

**Question.** Are rankings sensitive to a single year's observations?

I calculate 2020–2024 averages when the data supports them. A minimum of three non-missing annual observations is required per entity and indicator, reducing the effect of a single-year gap while retaining transparent coverage rules.

In [4]:
avg_rows = []
period = country_df.loc[country_df["year"].between(2020, 2024)]
for metric in bench_metrics:
    grouped = period.groupby(["country", "iso_code"])[metric].agg(["mean", "count"]).reset_index()
    grouped = grouped.loc[grouped["count"] >= 3].copy()
    if grouped.empty: continue
    grouped["indicator"] = metric
    grouped["period"] = "2020–2024; at least 3 observations"
    avg_rows.append(grouped)
multi_year = pd.concat(avg_rows, ignore_index=True) if avg_rows else pd.DataFrame()
multi_year.to_csv(TABLE_DIR / "country_multi_year_averages.csv", index=False)
display(multi_year.head(20))

,country,iso_code,mean,count,indicator,period
0,Afghanistan,AFG,43.864281,5,total_energy_twh,2020–2024; at least 3 observations
1,Albania,ALB,24.491238,5,total_energy_twh,2020–2024; at least 3 observations
2,Algeria,DZA,724.245230,5,total_energy_twh,2020–2024; at least 3 observations
3,American Samoa,ASM,1.946432,5,total_energy_twh,2020–2024; at least 3 observations
4,Angola,AGO,92.486094,5,total_energy_twh,2020–2024; at least 3 observations
5,Antarctica,ATA,0.050881,5,total_energy_twh,2020–2024; at least 3 observations
6,Antigua and Barbuda,ATG,3.044687,5,total_energy_twh,2020–2024; at least 3 observations
7,Argentina,ARG,888.235180,5,total_energy_twh,2020–2024; at least 3 observations
8,Armenia,ARM,47.658706,5,total_energy_twh,2020–2024; at least 3 observations
9,Aruba,ABW,4.828959,5,total_energy_twh,2020–2024; at least 3 observations


### Conclusion

A multi-year average can reduce noise from a single year, but it can also conceal recent structural changes. The average is only calculated where at least three years are observed; this rule does not guarantee that every country has identical reporting coverage. Compare the multi-year table with single-year rankings before describing a country as a stable leader or laggard.